# NUFROST Colab Launcher

> This is a launch script to load and run the NUFROST reconstruction algorithm in Google Colab environment.
You can open this notebook in Google Colab by right-clicking in Google Drive and selecting "Open with > Google Colaboratory".

## 1. Configuration Section
> configurable parameters for the NUFROST Colab Launcher.

In [ ]:
# NUFROST project path in Google Drive, do not include "MyDrive/".
PROJECT_PATH_IN_GDRIVE = "WorkSpaces/nufrost/"

# Data path in Google Drive, do not include "MyDrive/".
DATA_PATH_IN_GDRIVE = "WorkSpaces/nufrost/data/hls/"

# Parallel jobs, -1 means using all available cores.
N_JOBS = -1

# Image filename
# Target coordinates and band
TARGET_LON = -104.7915
TARGET_LAT = 39.8143
TARGET_BAND = "B02"

# Image filename. If empty, the script will automatically find matching chunks.
IMAGE_FILENAME = ""

# Output path in Google Drive, do not include "MyDrive/".
OUTPUT_PATH_IN_GDRIVE = "hants_out/"

# Cache directory path in colab.
CACHE_DIR_IN_COLAB = "/content/drive/MyDrive/cache/" # Also can put in Google Drive if needed.

# Mount point in Google Colab, usually do not change this.
MOUNT_POINT_IN_COLAB = "/content/drive"

# Target Time
TARGET_TIME = "2023-06-15 12:00:00"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive # type: ignore[import]
import os
from pathlib import Path
import sys

# mount Google Drive to colab
drive.mount(MOUNT_POINT_IN_COLAB)

PROJECT_ROOT_IN_COLAB = str(Path(MOUNT_POINT_IN_COLAB) / "MyDrive" / PROJECT_PATH_IN_GDRIVE)

# Check if the specified project root exists
if not Path(PROJECT_ROOT_IN_COLAB).exists():
    print(f"[Warning] Path not found: {PROJECT_ROOT_IN_COLAB}")
    print("Please make sure you have uploaded the code and modified the PROJECT_ROOT_IN_DRIVE variable above.")
else:
    # Add the project root directory to Python search path to enable importing src
    if PROJECT_ROOT_IN_COLAB not in sys.path:
        sys.path.append(PROJECT_ROOT_IN_COLAB)
    # Change working directory to the project root
    os.chdir(PROJECT_ROOT_IN_COLAB)
    print(f"[Success] Working directory changed to: {os.getcwd()}")

## 3. Install Dependencies
use `requirements.txt` to install necessary packages.

In [ ]:
%pip install -r requirements.txt

## 4. Run Reconstruction Task

Here we directly call the `src.reconstruct` interface for reconstruction.

In [ ]:
import src.data_loader
from src.data_loader import find_image_chunks
import os
import src
import importlib
from pathlib import Path

importlib.reload(src) # Ensure latest code is loaded

IMAGE_FOLDER = str(Path(MOUNT_POINT_IN_COLAB) / "MyDrive" / DATA_PATH_IN_GDRIVE)
OUTPUT_DIR_PATH = Path(MOUNT_POINT_IN_COLAB) / "MyDrive" / OUTPUT_PATH_IN_GDRIVE
OUTPUT_DIR_PATH.mkdir(parents=True, exist_ok=True)

if IMAGE_FILENAME:
    image_paths_list = [[str(Path(IMAGE_FOLDER) / IMAGE_FILENAME)]]
else:
    image_paths_list = find_image_chunks(
        data_dir=IMAGE_FOLDER,
        lon=TARGET_LON,
        lat=TARGET_LAT,
        band=TARGET_BAND
    )

for image_paths in image_paths_list:
    first_path = Path(image_paths[0])
    OUTPUT_PATH = str(OUTPUT_DIR_PATH / f"{first_path.name}_{TARGET_TIME}_recon_result.tif")

    if Path(OUTPUT_PATH).exists():
        print(f"Skipping {first_path.name}, output already exists.")
        continue

    src.reconstruct_nufrost(
        image=image_paths,
        target_time=TARGET_TIME,
        output_path=OUTPUT_PATH,
        n_jobs=N_JOBS,
        cache_dir=CACHE_DIR_IN_COLAB
    )

importlib.reload(src.data_loader)
